# Judge distillation — QLoRA on Qwen3-8B (Phase 4, run #2: fully resumable)

Trains the critique-judge task ("is this candidate finding genuine?") into Qwen3-8B,
so the pass can drop from the 35B to the 8B in production.

**Runtime:** Colab GPU (T4 16GB is enough). `Runtime → Change runtime type → T4 GPU`.

**Crash safety:** every long step persists to Drive and resumes where it left off.
If the runtime dies at ANY point, just rerun all cells top-to-bottom:

- data: uploaded once, then read from Drive (no re-upload prompt)
- training: checkpoints to Drive every 25 steps, auto-resume (max ~20 min lost)
- adapter: saved to Drive BEFORE eval
- eval: per-row predictions stream to Drive, finished rows are skipped on rerun

Only `pip install` + model download (~5-8 min total) rerun each session — those
cannot be usefully cached.

Data format: each row is `{"messages": [user, assistant]}` where the user turn is the
PRODUCTION judge prompt and the assistant turn is rationale-first JSON. Training masks
the user turn (loss on the response only). Chat template runs with thinking disabled,
matching production (`think=False`).


In [ ]:
%pip install -q unsloth
import torch
print(torch.cuda.get_device_name(0))


In [ ]:
# Mount Drive — everything below persists here so a dead runtime costs minutes,
# not hours.
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
OUT_DIR = DRIVE + '/judge-out'


In [ ]:
# Data: first session uploads and stashes to Drive; later sessions read from
# Drive and skip the upload prompt entirely.
import os, shutil

def ensure_data(name):
    drive_copy = f'{DRIVE}/{name}'
    if os.path.exists(drive_copy):
        shutil.copy(drive_copy, name)
        print(f'{name}: restored from Drive')
        return
    if os.path.exists(name):
        # already in this runtime (e.g. uploaded earlier in the session)
        shutil.copy(name, drive_copy)
        print(f'{name}: found locally, stashed to Drive')
        return
    from google.colab import files
    up = files.upload()
    assert up, f'upload {name}'
    # Colab renames clashing uploads to "name (1).jsonl" - accept whatever arrived
    got = name if name in up else next(iter(up))
    if got != name:
        shutil.move(got, name)
    shutil.copy(name, drive_copy)
    print(f'{name}: uploaded and stashed to Drive')

ensure_data('train.jsonl')
ensure_data('eval.jsonl')


In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 4096  # longest prompt ~2.7k tokens + completion
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3-8B-unsloth-bnb-4bit',  # fallback: 'unsloth/Qwen3-8B'
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=41,
)

In [ ]:
import json
from datasets import Dataset

def load_rows(path):
    return [json.loads(l) for l in open(path, encoding='utf-8')]

def to_text(row):
    # thinking disabled to match production (Ollama think=False)
    return tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False,
        enable_thinking=False,
    )

train_rows = load_rows('train.jsonl')
train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_rows])
print(len(train_ds), 'train examples')
print(train_ds[0]['text'][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import glob

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type='linear',
        logging_steps=10,
        optim='adamw_8bit',
        seed=41,
        output_dir=OUT_DIR,
        save_strategy='steps',
        save_steps=25,
        save_total_limit=2,
        report_to='none',
    ),
)
# loss on assistant turns only
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# Resume from the last Drive checkpoint if one exists (crash recovery).
has_ckpt = bool(glob.glob(OUT_DIR + '/checkpoint-*'))
print('resuming from checkpoint' if has_ckpt else 'fresh run')
trainer.train(resume_from_checkpoint=has_ckpt)


In [ ]:
# SAVE FIRST, EVAL LATER: secure the adapter before the long eval loop,
# so losing the session mid-eval only costs the eval.
model.save_pretrained('judge-adapter')
tokenizer.save_pretrained('judge-adapter')
!zip -qr judge-adapter.zip judge-adapter
!cp judge-adapter.zip /content/drive/MyDrive/
print('saved: /content/drive/MyDrive/judge-adapter.zip')
from google.colab import files
files.download('judge-adapter.zip')


In [ ]:
# Evaluation: per-category precision/recall on eval.jsonl.
# RESUMABLE: each row's prediction is appended to Drive as soon as it's made;
# on rerun, already-judged rows are skipped and the table is rebuilt from the file.
# Set EVAL_BASE=True on a fresh runtime (before training) for the untuned baseline
# (predictions then go to a separate file so runs don't mix).
import re, collections, os
EVAL_BASE = False

PRED_PATH = f'{OUT_DIR}/eval-preds{"-base" if EVAL_BASE else ""}.jsonl'
os.makedirs(OUT_DIR, exist_ok=True)

FastLanguageModel.for_inference(model)
eval_rows = load_rows('eval.jsonl')

def judge(user_content):
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    ids = tokenizer(text, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=160, temperature=0.1, do_sample=False)
    reply = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'"genuine"\s*:\s*(true|false)', reply)
    return (m.group(1) == 'true') if m else None, reply

def category_of(user_content):
    m = re.search(r'Proposed finding \(([^)]+)\)', user_content)
    return m.group(1) if m else '?'

# resume: load already-judged rows
done = {}
if os.path.exists(PRED_PATH):
    for line in open(PRED_PATH, encoding='utf-8'):
        rec = json.loads(line)
        done[rec['i']] = rec
    print(f'resuming eval: {len(done)}/{len(eval_rows)} rows already judged')

pred_f = open(PRED_PATH, 'a', encoding='utf-8')
for i, row in enumerate(eval_rows):
    if i in done:
        continue
    user = row['messages'][0]['content']
    gold = json.loads(re.sub(r'^```json\n|\n```$', '', row['messages'][1]['content']))['genuine']
    pred, reply = judge(user)
    rec = {'i': i, 'cat': category_of(user), 'gold': gold, 'pred': pred, 'reply': reply}
    pred_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    pred_f.flush()
    done[i] = rec
    if (i+1) % 25 == 0: print(f'{i+1}/{len(eval_rows)}')
pred_f.close()

# table from the full prediction file
stats = collections.defaultdict(lambda: {'tp':0,'fp':0,'fn':0,'tn':0,'unparsed':0})
for rec in done.values():
    for key in (rec['cat'], 'TOTAL'):
        s = stats[key]
        if rec['pred'] is None: s['unparsed'] += 1
        elif rec['pred'] and rec['gold']: s['tp'] += 1
        elif rec['pred'] and not rec['gold']: s['fp'] += 1
        elif not rec['pred'] and rec['gold']: s['fn'] += 1
        else: s['tn'] += 1

print(f"\n{'category':28s} {'P':>6s} {'R':>6s} {'n':>5s} {'unparsed':>9s}")
for cat, s in sorted(stats.items()):
    n = s['tp']+s['fp']+s['fn']+s['tn']+s['unparsed']
    p = s['tp']/max(s['tp']+s['fp'],1); r = s['tp']/max(s['tp']+s['fn'],1)
    print(f'{cat:28s} {p:6.2f} {r:6.2f} {n:5d} {s["unparsed"]:9d}')


## Optional: merged GGUF for Ollama

Run the next cell only if eval numbers justify deployment. It merges the adapter
and exports a q4_k_m GGUF (~5GB download), then serve locally with:

```
# Modelfile
FROM ./judge-q4_k_m.gguf
```
`ollama create qwen3-8b-judge -f Modelfile` — then point the audit role's
fast-model env at `qwen3-8b-judge` for the critique judge and re-run
`micro_runner --pass critique` on the dev set before trusting it.

In [ ]:
# model.save_pretrained_gguf('judge-gguf', tokenizer, quantization_method='q4_k_m')
# files.download('judge-gguf/unsloth.Q4_K_M.gguf')